# Reading and Writing Data

{doc}`Introduction to DataFrames <01_introduction_to_dataframes>` built
DataFrames by hand. Real pipelines read them from storage and write their
results back. This page covers where data lives on Databricks, how to read
tables and files, and how to write results out safely.

## Where Data Lives on Databricks

On Databricks, data is governed by **Unity Catalog**, which addresses tables,
views and volumes with a three-level namespace, `catalog.schema.object`
([What is Unity Catalog?](https://docs.databricks.com/aws/en/data-governance/unity-catalog/)):

```mermaid
:caption: The part of the Unity Catalog hierarchy this page uses.
:align: center

graph TD
    C[Catalog] --> S[Schema]
    S --> T[Tables and views]
    S --> V[Volumes]
    T --> TN["catalog.schema.table"]
    V --> VP["/Volumes/catalog/schema/volume/..."]
```

- **Tables** hold structured data and are addressed as
  `catalog.schema.table`. Unless otherwise specified, all tables on Databricks
  are **Delta Lake** tables
  ([What is Delta Lake?](https://docs.databricks.com/aws/en/delta/)). Delta
  Lake extends Parquet data files with a transaction log that provides ACID
  transactions
  ([What is Delta Lake?](https://docs.databricks.com/aws/en/delta/)).
  Unity Catalog *managed* tables, where Unity Catalog also manages the
  underlying storage, are the default and recommended table type
  ([Unity Catalog managed tables](https://docs.databricks.com/aws/en/tables/managed)).
- **Volumes** govern non-tabular data of any format and are addressed as
  paths of the form `/Volumes/<catalog>/<schema>/<volume>/<path>`, whether you
  use Spark, SQL or plain Python
  ([What are Unity Catalog volumes?](https://docs.databricks.com/aws/en/volumes/)).
  Use them for raw files you ingest, or files you need to hand to other
  systems.

The older Databricks File System (DBFS) root and DBFS mounts are deprecated;
Databricks recommends Unity Catalog volumes, external locations or workspace
files instead
([What is DBFS?](https://docs.databricks.com/aws/en/dbfs/)).

:::{admonition} Note: The Catalog and Schema Used on This Page
:class: note

Unity Catalog provides sample datasets in the `samples` catalog
([Sample datasets](https://docs.databricks.com/aws/en/discover/databricks-datasets#unity-catalog-datasets));
this page reads from `samples.nyctaxi.trips`. To write, you need a schema
where you can create tables and volumes. On the course's Free Edition
workspace the `workspace.default` schema works, as the cells below show.
Substitute your own catalog and schema if you are elsewhere.
:::

The page creates one volume and two tables, all prefixed `readwrite02_`. The
first cell drops them, so the notebook can be re-run from the top:

In [1]:
from pyspark.sql import functions as F

from project.utils.sparksession import get_spark_session

spark = get_spark_session(profile="free-edition")

for table in ["readwrite02_trips_bronze", "readwrite02_long_trips"]:
    spark.sql(f"DROP TABLE IF EXISTS workspace.default.{table}")
spark.sql("DROP VOLUME IF EXISTS workspace.default.readwrite02_raw");

## The DataFrameReader

All reads start from `spark.read`, a `DataFrameReader`. The general pattern
is:

```python
df = (
    spark.read
    .format("<format>")          # csv, json, parquet, delta, text, ...
    .option("<key>", "<value>")  # format-specific options
    .schema("<ddl schema>")      # optional: declare the schema up front
    .load("<path>")
)
```

Each common format also has a shortcut such as `spark.read.csv(path, ...)`,
`spark.read.json(path)` or `spark.read.parquet(path)`.

### Reading Tables

Reading a table needs only its name. `spark.read.table(...)` and
`spark.table(...)` are equivalent:

In [2]:
trips = spark.read.table("samples.nyctaxi.trips")
trips.printSchema()

root
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- pickup_zip: integer (nullable = true)
 |-- dropoff_zip: integer (nullable = true)



The schema comes from the catalog, so there is nothing to infer and nothing
to declare.

### Reading Files from a Volume

Let's create a volume and put a CSV extract of the taxi data in it, so there
is a file to read:

In [3]:
spark.sql("CREATE VOLUME workspace.default.readwrite02_raw")

(
    trips
    .limit(1000)
    .repartition(4)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv("/Volumes/workspace/default/readwrite02_raw/trips_csv")
)

Spark writes a **directory** of part files, one per partition of the
DataFrame, rather than a single file. The extract above was split into 4
partitions, and the hidden `_metadata` column (covered
{ref}`below <readwrite-metadata>`) shows the 4 files that came out. Readers
accept the directory path and read every file in it:

In [4]:
(
    spark.read.option("header", True)
    .csv("/Volumes/workspace/default/readwrite02_raw/trips_csv")
    .select(F.col("_metadata.file_name"))
    .distinct()
    .show(truncate=False)
)

+--------------------------------------------------------------------------------------+
|file_name                                                                             |
+--------------------------------------------------------------------------------------+
|part-00000-tid-7433085069270178696-4eb4222d-dfb6-453f-8d07-9c26804a6121-306-1-c000.csv|
|part-00002-tid-7433085069270178696-4eb4222d-dfb6-453f-8d07-9c26804a6121-308-1-c000.csv|
|part-00001-tid-7433085069270178696-4eb4222d-dfb6-453f-8d07-9c26804a6121-307-1-c000.csv|
|part-00003-tid-7433085069270178696-4eb4222d-dfb6-453f-8d07-9c26804a6121-309-1-c000.csv|
+--------------------------------------------------------------------------------------+



Reading the CSV back with schema inference:

In [5]:
trips_inferred = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/default/readwrite02_raw/trips_csv")
)
trips_inferred.printSchema()

root
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- pickup_zip: integer (nullable = true)
 |-- dropoff_zip: integer (nullable = true)



Schema inference is convenient for exploration, but the result depends on
the data: a column that happens to contain only digits today becomes an
`int`, then breaks when a letter shows up tomorrow. When the schema is known,
specify it for the CSV reader
([CSV file](https://docs.databricks.com/aws/en/query/formats/csv#specify-a-schema)):

In [6]:
trips_schema = """
    tpep_pickup_datetime TIMESTAMP,
    tpep_dropoff_datetime TIMESTAMP,
    trip_distance DOUBLE,
    fare_amount DOUBLE,
    pickup_zip INT,
    dropoff_zip INT
"""

trips_csv = (
    spark.read
    .schema(trips_schema)
    .option("header", True)
    .csv("/Volumes/workspace/default/readwrite02_raw/trips_csv")
)
trips_csv.count()

1000

Some useful options for CSV files. The `mode` values are listed under
[Handle malformed CSV records](https://docs.databricks.com/aws/en/query/formats/csv#handle-malformed-csv-records);
the full list of reader options is in
[DataFrameReader options](https://docs.databricks.com/aws/en/spark/api-options#batch-read-options).

| Option | Meaning |
| --- | --- |
| `header` | First line contains column names |
| `sep` | Field delimiter |
| `inferSchema` | Guess column types from the data |
| `multiLine` | Records may span several lines |
| `mode` | Malformed rows: `PERMISSIVE` (default, unparsable fields become null), `DROPMALFORMED` (drop the line) or `FAILFAST` (abort the read) |
| `rescuedDataColumn` | Put fields that don't match the schema into a JSON column instead of losing them |

Self-describing formats such as **Parquet** and **Delta** store their schema
with the data, so they need neither inference nor a declared schema.

(readwrite-metadata)=
### Which File Did a Row Come From?

Every file-based DataFrame has a hidden `_metadata` column, a struct with
fields such as `file_path`, `file_name`, `file_size` and
`file_modification_time`. To use it, select it explicitly
([File metadata column](https://docs.databricks.com/aws/en/ingestion/file-metadata-column)):

In [7]:
trips_csv.select("trip_distance", "_metadata.file_name", "_metadata.file_size").show(3, truncate=False)

+-------------+--------------------------------------------------------------------------------------+---------+
|trip_distance|file_name                                                                             |file_size|
+-------------+--------------------------------------------------------------------------------------+---------+
|0.6          |part-00000-tid-7433085069270178696-4eb4222d-dfb6-453f-8d07-9c26804a6121-306-1-c000.csv|17841    |
|1.24         |part-00000-tid-7433085069270178696-4eb4222d-dfb6-453f-8d07-9c26804a6121-306-1-c000.csv|17841    |
|0.9          |part-00000-tid-7433085069270178696-4eb4222d-dfb6-453f-8d07-9c26804a6121-306-1-c000.csv|17841    |
+-------------+--------------------------------------------------------------------------------------+---------+
only showing top 3 rows


This replaces the older `input_file_name()` function, which is not available
on Unity Catalog, is deprecated in Databricks Runtime 13.3 LTS and above, and
is no longer supported in Databricks Runtime 17.3 LTS and above
([input_file_name function](https://docs.databricks.com/aws/en/sql/language-manual/functions/input_file_name)).

### Reading Files with SQL

In SQL, the `read_files` table-valued function reads the files under a
location and can infer a unified schema across all of them. Its options are
passed as named parameters, such as `format => 'csv'`
([read_files table-valued function](https://docs.databricks.com/aws/en/sql/language-manual/functions/read_files)):

In [8]:
spark.sql("""
    SELECT count(*) AS n
    FROM read_files(
        '/Volumes/workspace/default/readwrite02_raw/trips_csv',
        format => 'csv',
        header => true
    )
""").show()

+----+
|   n|
+----+
|1000|
+----+



## The DataFrameWriter

Writes start from `df.write`, a `DataFrameWriter`, and mirror the reader:

```python
(
    df.write
    .format("<format>")       # Delta unless you say otherwise, for tables
    .mode("<save mode>")      # what to do if the target exists
    .option("<key>", "<value>")
    .saveAsTable("<catalog.schema.table>")   # or .save("<path>") for files
)
```

### Save Modes

The **save mode** decides what happens when the target already exists
([Apache Spark: Generic Load/Save Functions](https://spark.apache.org/docs/latest/sql-data-sources-load-save-functions.html#save-modes)):

| Mode | Behaviour if the target exists |
| --- | --- |
| `errorifexists` (default) | Fail with an error |
| `append` | Add the new rows to the existing data |
| `overwrite` | Replace the existing data |
| `ignore` | Leave the existing data unchanged and write nothing |

### Writing Tables

Writing a managed Delta table to Unity Catalog:

In [9]:
(
    trips_csv.write
    .mode("overwrite")
    .saveAsTable("workspace.default.readwrite02_trips_bronze")
)

spark.sql("SELECT count(*) AS n FROM workspace.default.readwrite02_trips_bronze").show()

+----+
|   n|
+----+
|1000|
+----+



The table is immediately queryable by name, from SQL and from any other
notebook with access to it.

Because the table is in Delta format, every write is a transaction that
either succeeds or fails completely: data files from a failed write never
become part of the table
([ACID guarantees on Databricks](https://docs.databricks.com/aws/en/lakehouse/acid#atomicity)).
Delta also enforces the table's schema on write
([Schema enforcement](https://docs.databricks.com/aws/en/delta/schema-validation)).
Appending a DataFrame with a column the table doesn't have fails:

In [10]:
(
    trips_csv
    .withColumn("source", F.lit("csv"))
    .write
    .mode("append")
    .saveAsTable("workspace.default.readwrite02_trips_bronze")
)

AnalysisException: [DELTA_METADATA_MISMATCH] A metadata mismatch was detected when writing to the Delta table.
- A schema mismatch detected when writing to the Delta table (Table ID: 512add35-28b3-4fb4-9cab-4234e0e0ba6d).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set: '.option("mergeSchema", "true")'.
For other operations, set the session configuration spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation specific to the operation for details.

Table schema:
root
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- pickup_zip: integer (nullable = true)
 |-- dropoff_zip: integer (nullable = true)


Data schema:
root
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- far

The `AnalysisException` (`DELTA_METADATA_MISMATCH`) prints the table's schema
next to the schema of the data being written. To let the schema evolve on
purpose, set the `mergeSchema` option on the write, as the message itself
suggests
([Schema enforcement](https://docs.databricks.com/aws/en/delta/schema-validation#modify-a-table-schema)).

An alternative is the `DataFrameWriterV2` API, reached through
`df.writeTo(...)`, which creates, replaces or appends to tables with explicit
verbs
([Apache Spark API reference: DataFrame.writeTo](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.writeTo.html)):

```python
trips_csv.writeTo("workspace.default.readwrite02_trips_bronze").createOrReplace()
trips_csv.writeTo("workspace.default.readwrite02_trips_bronze").append()
```

### Writing Files

To produce files, for example to hand data to another system, use `save` (or
a format shortcut) with a volume path:

In [11]:
(
    trips_csv.write
    .mode("overwrite")
    .parquet("/Volumes/workspace/default/readwrite02_raw/trips_parquet")
)
spark.read.parquet("/Volumes/workspace/default/readwrite02_raw/trips_parquet").count()

1000

`partitionBy` splits the output into one subdirectory per value of the given
columns, which is why it has limited applicability to columns with high
cardinality
([Apache Spark: Generic Load/Save Functions](https://spark.apache.org/docs/latest/sql-data-sources-load-save-functions.html#bucketing-sorting-and-partitioning)).
Here each pickup date becomes a `pickup_date=...` directory, visible in the
file paths:

In [12]:
(
    trips_csv
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .write
    .mode("overwrite")
    .partitionBy("pickup_date")
    .parquet("/Volumes/workspace/default/readwrite02_raw/trips_by_date")
)

(
    spark.read.parquet("/Volumes/workspace/default/readwrite02_raw/trips_by_date")
    .select(F.regexp_extract("_metadata.file_path", r"(pickup_date=[^/]+)", 1).alias("directory"))
    .distinct()
    .orderBy("directory")
    .show(5, truncate=False)
)

+----------------------+
|directory             |
+----------------------+
|pickup_date=2016-02-01|
|pickup_date=2016-02-02|
|pickup_date=2016-02-03|
|pickup_date=2016-02-04|
|pickup_date=2016-02-05|
+----------------------+
only showing top 5 rows


For Delta tables on Databricks, don't hand-pick partitions. Databricks
recommends liquid clustering for all new tables, which replaces table
partitioning and `ZORDER`; enable it with `CLUSTER BY` in `CREATE TABLE` or
the DataFrame APIs
([Use liquid clustering for tables](https://docs.databricks.com/aws/en/tables/clustering)).
{doc}`Using Apache Spark with Delta Lake <../../04_monitoring_and_optimizing_spark_workloads_on_databricks/02_using_apache_spark_with_delta_lake>`
covers it in detail.

:::::{dropdown} Example: Ingest a CSV Extract into a Table
:open:
:color: primary
:icon: pencil
:class-body: example-body

We want a table of long taxi trips, built from the CSV extract in the
`readwrite02_raw` volume, with a declared schema and the source file recorded
on each row.

**Read with an explicit schema.**

```python
raw = (
    spark.read
    .schema(trips_schema)
    .option("header", True)
    .csv("/Volumes/workspace/default/readwrite02_raw/trips_csv")
)
```

**Keep the long trips and record the source file.**

```python
long_trips = (
    raw
    .filter(F.col("trip_distance") > 10)
    .withColumn("source_file", F.col("_metadata.file_path"))
)
```

**Write the result as a managed Delta table.**

```python
long_trips.write.mode("overwrite").saveAsTable("workspace.default.readwrite02_long_trips")
```

Only the final `saveAsTable` is an action, so Spark plans and runs the read,
filter and write together. The cell below runs these steps and counts the
rows that landed in the table.
:::::

In [13]:
raw = (
    spark.read
    .schema(trips_schema)
    .option("header", True)
    .csv("/Volumes/workspace/default/readwrite02_raw/trips_csv")
)

long_trips = (
    raw
    .filter(F.col("trip_distance") > 10)
    .withColumn("source_file", F.col("_metadata.file_path"))
)

long_trips.write.mode("overwrite").saveAsTable("workspace.default.readwrite02_long_trips")

spark.read.table("workspace.default.readwrite02_long_trips").groupBy("source_file").count().show(truncate=False)

+------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|source_file                                                                                                                                     |count|
+------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|dbfs:/Volumes/workspace/default/readwrite02_raw/trips_csv/part-00000-tid-7433085069270178696-4eb4222d-dfb6-453f-8d07-9c26804a6121-306-1-c000.csv|17   |
|dbfs:/Volumes/workspace/default/readwrite02_raw/trips_csv/part-00001-tid-7433085069270178696-4eb4222d-dfb6-453f-8d07-9c26804a6121-307-1-c000.csv|12   |
|dbfs:/Volumes/workspace/default/readwrite02_raw/trips_csv/part-00002-tid-7433085069270178696-4eb4222d-dfb6-453f-8d07-9c26804a6121-308-1-c000.csv|18   |
|dbfs:/Volumes/workspace/default/readwrite02_raw/trips_csv/part-00003-tid-74330850

The `source_file` values carry a `dbfs:/` prefix in front of the `/Volumes`
path; Spark accepts volume paths with or without it
([What are Unity Catalog volumes?](https://docs.databricks.com/aws/en/volumes/)).

## When to Use Tables and When to Use Files

| Store as | Use it for | Read and write with |
| --- | --- | --- |
| Managed Delta table | Anything you query again: pipeline layers, analytics, results | `spark.read.table`, `saveAsTable`, `writeTo` |
| Files in a volume | Raw files arriving from other systems, and exports to them | `spark.read.<format>(path)`, `df.write.<format>(path)`, `read_files` |

Managed tables are the default and recommended table type, and Unity Catalog
handles their storage and optimisation
([Unity Catalog managed tables](https://docs.databricks.com/aws/en/tables/managed)).
Volumes are for non-tabular data, and files in a volume can't be registered as
Unity Catalog tables
([What are Unity Catalog volumes?](https://docs.databricks.com/aws/en/volumes/)).

## Common Mistakes

- **Re-running a write without a save mode.** The default mode is
  `errorifexists`
  ([Apache Spark: Generic Load/Save Functions](https://spark.apache.org/docs/latest/sql-data-sources-load-save-functions.html#save-modes)),
  so a notebook cell that wrote a table once fails the second time:

In [14]:
trips_csv.write.saveAsTable("workspace.default.readwrite02_trips_bronze")

AnalysisException: [TABLE_OR_VIEW_ALREADY_EXISTS] Cannot create table or view `default`.`readwrite02_trips_bronze` because it already exists.
Choose a different name, drop the existing object, add the IF NOT EXISTS clause to tolerate pre-existing objects, add the OR REPLACE clause to replace the existing materialized view, or add the OR REFRESH clause to refresh the existing streaming table. SQLSTATE: 42P07

JVM stacktrace:
org.apache.spark.sql.catalyst.analysis.TableAlreadyExistsException
	at org.apache.spark.sql.errors.QueryCompilationErrors$.tableAlreadyExistsError(QueryCompilationErrors.scala:3344)
	at org.apache.spark.sql.execution.datasources.v2.AtomicCreateTableAsSelectExec.run(WriteToDataSourceV2Exec.scala:225)
	at org.apache.spark.sql.execution.datasources.v2.V2CommandExec.$anonfun$result$3(V2CommandExec.scala:55)
	at org.apache.spark.sql.execution.SparkPlan.runCommandInAetherOrSpark(SparkPlan.scala:231)
	at org.apache.spark.sql.execution.datasources.v2.V2CommandExec.$anonfun$

  The error is `TABLE_OR_VIEW_ALREADY_EXISTS`. Choose the mode on purpose:
  `overwrite` for a full refresh, `append` to add rows.
- **Relying on `inferSchema` in a pipeline.** Inferred types change when the
  data does. Specify the schema when you know it
  ([CSV file](https://docs.databricks.com/aws/en/query/formats/csv#specify-a-schema)).
- **Partitioning small tables, or by a high-cardinality column.** Databricks
  advises not to partition tables with less than 1 TB of data, to keep each
  partition at 1 GB or more, and notes that partitioning works well only for
  low or known cardinality fields
  ([When to partition tables on Databricks](https://docs.databricks.com/aws/en/tables/partitions#minimum-size-recommendations)).
  Use liquid clustering instead.
- **Storing data in DBFS root or mounts.** Both are deprecated; use volumes
  and tables
  ([What is DBFS?](https://docs.databricks.com/aws/en/dbfs/)).
- **Using `input_file_name()`.** It isn't available on Unity Catalog. Use the
  `_metadata` column
  ([input_file_name function](https://docs.databricks.com/aws/en/sql/language-manual/functions/input_file_name)).

## Cleaning Up

Tables and volumes are persistent. Drop the ones you created on this page
when you no longer need them:

In [15]:
for table in ["readwrite02_trips_bronze", "readwrite02_long_trips"]:
    spark.sql(f"DROP TABLE IF EXISTS workspace.default.{table}")
spark.sql("DROP VOLUME IF EXISTS workspace.default.readwrite02_raw");

Dropping a managed table starts a recovery period (7 days by default), during
which `UNDROP TABLE` can restore it; after that Databricks deletes the data
files
([Unity Catalog managed tables](https://docs.databricks.com/aws/en/tables/managed#drop-a-managed-table)).
Dropping a managed volume works the same way, with 7-day retention
([What are Unity Catalog volumes?](https://docs.databricks.com/aws/en/volumes/)).

With reading, transforming and writing covered, the next section looks at
what happens on the cluster while these operations run:
{doc}`Distributed Systems Programming Fundamentals <../03_distributed_systems_programming_fundamentals/index>`.

## Resources

- [Query data (Databricks)](https://docs.databricks.com/aws/en/query/)
- [What are Unity Catalog volumes? (Databricks)](https://docs.databricks.com/aws/en/volumes/)
- [Data sources (Apache Spark)](https://spark.apache.org/docs/latest/sql-data-sources.html)